# 📈 StockCast — Stock Price Prediction (ML Minor Project)

Guided end-to-end walkthrough: data → EDA → feature engineering → LSTM +
classical baselines → evaluation → forward forecast.

The heavy lifting lives in the `src/` package; this notebook just drives it
and visualises the results. Run every cell top-to-bottom.

In [ ]:
import sys
from pathlib import Path
from IPython.display import display, Image

# make the project package importable when running from notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
from src import config, data, eda, preprocessing, pipeline, predict, utils

utils.set_seed()
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')
print('Project root:', ROOT)

## 1. Load / build the dataset

Runs fully offline via a seeded Geometric-Brownian-Motion OHLCV series.
To use real Kaggle/Yahoo data, replace `data/raw/AAPL.csv` (see `README.md`).

In [ ]:
df = data.build_dataset()
print(f'{len(df)} rows | {df.index.min.date()} -> {df.index.max.date()}')
df.tail()

## 2. Exploratory Data Analysis

In [ ]:
features = preprocessing.add_features(df)
summary = eda.run_eda(df, features)

for fig in ['01_price_volume.png', '02_returns.png', '03_correlation.png']:
    display(Image(filename=str(config.FIGURES_DIR / fig)))

## 3. Train & evaluate all models

`run_pipeline` does a leakage-safe chronological split, fits Ridge / Random
Forest / Gradient Boosting and the PyTorch LSTM, and saves every figure and
metric into `reports/`.

In [ ]:
results = pipeline.run_pipeline(verbose=True)

## 4. Results

In [ ]:
metrics = pd.DataFrame(results['metrics']).T.sort_values('RMSE')
direction = pd.DataFrame(results['direction']).T[['direction_accuracy_%']]
print('Best model (by RMSE):', results['best_model'])
print('--- Price-level error ---'); display(metrics)
print('--- Directional accuracy (%) ---');
display(direction.sort_values('direction_accuracy_%', ascending=False))

In [ ]:
for fig in ['04_predictions.png', '05_lstm_loss.png',
            '06_metrics.png', '07_feature_importance.png']:
    display(Image(filename=str(config.FIGURES_DIR / fig)))

## 5. Forward forecast

In [ ]:
fc = pd.DataFrame(predict.forecast_next_day(steps=5))
display(fc)

## 6. Takeaways

- Forecast target is the **next-day return** (stationary), then price is
  reconstructed as `prev_close * (1 + r)` - this keeps tree R² healthy.
- **LSTM** forecasts the price **level** best (R² ~ 0.98, MAPE ~ 1.1%),
  essentially matching the naive random-walk benchmark.
- **Day-ahead direction is near chance (46-48%)** for all models - an honest
  random-walk / efficient-market result, not a bug.
- Swap in any real Kaggle/Yahoo CSV with zero code changes.